# Shared — `toy_sessions`

Tiny shared clickstream for Journey algorithm examples.

Companion script: `toy_sessions.py`

**Run with cwd = repo root** (or the script's folder / `examples`).


## Setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

HERE = Path.cwd()
if (HERE / "day-1").exists():
    _DAY1_ROOT = HERE / "day-1"
elif HERE.name in {"01_sessionize", "02_journey", "03_state_modeling", "04_feature_engineering", "05_segmentation", "_shared"}:
    _DAY1_ROOT = HERE.parent
elif HERE.name == "examples":
    _DAY1_ROOT = HERE.parents[1]
else:
    _DAY1_ROOT = HERE

# Make Path(__file__) match the companion script (same as running the .py).
__file__ = str(_DAY1_ROOT / "_shared/toy_sessions.py")
print("DAY1 root =", _DAY1_ROOT)
print("__file__ =", __file__)


## Imports & constants


In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path(__file__).resolve().parent / "data"
TOY_SESSIONS_CSV = DATA / "toy_sessions.csv"

# Event → stage (same ladder as Day 1 journey analysis).
STAGE_BY_EVENT = {
    "page_view": "landing",
    "scroll": "scroll",
    "engaged_read": "engaged",
    "subscription_view": "intent",
    "subscribe_click": "intent",
    "checkout_start": "deep",
    "payment_success": "deep",
    "register_complete": "registration",
    "subscription_active": "subscription",
}

FUNNEL_STAGES = [
    "landing",
    "scroll",
    "engaged",
    "intent",
    "deep",
    "registration",
    "subscription",
]


## Helpers (one function per cell)


### `load_toy_events`


In [ ]:
def load_toy_events() -> pd.DataFrame:
    """Return a mini sessionized event table with stage tags."""
    if not TOY_SESSIONS_CSV.exists():
        raise FileNotFoundError(
            f"missing {TOY_SESSIONS_CSV}. Run: python day-1/_shared/generate_toy_data.py"
        )
    frame = pd.read_csv(TOY_SESSIONS_CSV)
    frame["timestamp"] = pd.to_datetime(frame["timestamp"])
    if "anonymous_id" not in frame.columns:
        frame["anonymous_id"] = frame["session_idx"].map(lambda s: f"A{int(s):04d}")
    if "stage" not in frame.columns:
        frame["stage"] = frame["event_name"].map(STAGE_BY_EVENT)
    else:
        # Keep CSV stage, but fill gaps from the map.
        missing = frame["stage"].isna()
        if missing.any():
            frame.loc[missing, "stage"] = frame.loc[missing, "event_name"].map(STAGE_BY_EVENT)
    return frame


### `describe_sessions`


In [ ]:
def describe_sessions() -> None:
    print("Toy sessions (4 visits)")
    print(f"  source: {TOY_SESSIONS_CSV.relative_to(DATA.parent.parent)}")
    print("  S1: page_view                         → bounce")
    print("  S2: page_view → scroll → engaged_read → engaged exit")
    print("  S3: … → subscription_view → subscribe_click → intent exit")
    print("  S4: … → checkout → payment → subscription_active → convert")


## Run


In [ ]:
describe_sessions()
